# Register your own X-ray to your own CT

This notebook runs [`xvr`](https://github.com/eigenvivek/xvr) on data you provide. `xvr` is the reference implementation of Gopalakrishnan and colleagues, *Rapid patient-specific neural networks for X-ray to volume registration*, Nature 2026 ([doi:10.1038/s41586-026-11045-x](https://doi.org/10.1038/s41586-026-11045-x)), MIT licensed. The differentiable renderer underneath it is [`nanodrr`](https://github.com/eigenvivek/nanodrr), by the same author.

The [browser demo](https://vanioantunes.com/3d/xvr/) this notebook accompanies shows the idea on a fixed CT. This runs the real thing on yours.

## Before you start

**Set a GPU runtime.** *Runtime → Change runtime type → T4 GPU*. It works on CPU, slowly.

**What you need ready.**

1. A CT or MR volume as NIfTI, `.nii` or `.nii.gz`. If you have a DICOM folder, cell 3 converts it.
2. One radiograph or fluoroscopy frame, as DICOM or as a 16-bit image.
3. The geometry of the machine that took the radiograph: source-to-detector distance in mm, detector pixel spacing in mm, and the image size in pixels. In a DICOM these are `DistanceSourceToDetector` (0018,1110), `ImagerPixelSpacing` (0018,1164) and `Rows`/`Columns`.

**Privacy.** Colab runs on Google's machines. Anything uploaded here leaves your computer. Do not put identifiable patient data in it without the approvals that requires. Research and teaching only; this is not a medical device and nothing here is validated for clinical use.

**Registration needs a starting guess.** `xvr register` is a local optimiser, exactly like the browser demo: it refines a pose, it does not find one from nowhere. Either give it an initial pose in roughly the right neighbourhood, or train a patient-specific network first with `xvr train` so the network supplies one. Both are below.

## 1. Check the runtime

In [ ]:
!nvidia-smi || echo "No GPU. Runtime > Change runtime type > T4 GPU, then run this cell again."

import sys
import torch

print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 2. Install xvr

One package, from PyPI. A minute or two on a fresh runtime.

In [ ]:
%pip install -q xvreg

!xvr --version

## 3. Upload your data

Run the cell and pick your files. Upload the CT (`.nii` or `.nii.gz`) and the radiograph together.

For anything larger than a few hundred megabytes, mount Drive instead: uncomment the Drive lines and put the files in a folder there. Browser uploads of large NIfTI files are slow and time out.

In [ ]:
from pathlib import Path

WORK = Path("/content/xvr-run")
WORK.mkdir(parents=True, exist_ok=True)

# Option A: upload from your computer.
from google.colab import files

uploaded = files.upload()
for name in uploaded:
    Path(name).rename(WORK / name)

# Option B: use Google Drive instead of uploading.
# from google.colab import drive
# drive.mount("/content/drive")
# WORK = Path("/content/drive/MyDrive/xvr-run")

for f in sorted(WORK.iterdir()):
    print(f"{f.name:45s} {f.stat().st_size / 1e6:8.1f} MB")

### If your CT is a DICOM folder

Zip the folder, upload the zip above, then run this cell to unpack and convert it to a single NIfTI.

In [ ]:
%pip install -q dicom2nifti

import zipfile

import dicom2nifti

DICOM_ZIP = WORK / "ct-dicom.zip"   # change to your file name

if DICOM_ZIP.exists():
    dicom_dir = WORK / "ct-dicom"
    with zipfile.ZipFile(DICOM_ZIP) as z:
        z.extractall(dicom_dir)
    out = WORK / "nifti"
    out.mkdir(exist_ok=True)
    dicom2nifti.convert_directory(str(dicom_dir), str(out), compression=True)
    for f in sorted(out.glob("*.nii.gz")):
        print("wrote", f)
else:
    print(f"No {DICOM_ZIP.name}; skip this cell if your CT is already NIfTI.")

## 4. Read the geometry out of the radiograph

If the radiograph is a DICOM, this pulls the three numbers `xvr` needs. If it is not, read them off the console of the machine that took it and fill them in by hand in cell 6.

In [ ]:
%pip install -q pydicom

import pydicom

XRAY = WORK / "xray.dcm"   # change to your file name

if XRAY.exists() and XRAY.suffix.lower() == ".dcm":
    ds = pydicom.dcmread(str(XRAY))
    print("source to detector (mm):", getattr(ds, "DistanceSourceToDetector", "not in header"))
    print("source to patient (mm): ", getattr(ds, "DistanceSourceToPatient", "not in header"))
    print("pixel spacing (mm):     ", getattr(ds, "ImagerPixelSpacing", getattr(ds, "PixelSpacing", "not in header")))
    print("size (rows x cols):     ", ds.Rows, "x", ds.Columns)
else:
    print("Not a DICOM, or the name above is wrong. Fill the numbers in by hand in cell 6.")

## 5. What the CLI expects

Flag names change between `xvr` releases, so read them from the version you just installed rather than trusting a notebook written months ago. Run this, then build the command in the next cell from what it prints.

In [ ]:
!xvr register --help

## 6. Register

Set your file names and geometry, then run. If `--help` above names the arguments differently, edit the command to match.

Preprocessing matters more than people expect. A DRR is bright where the patient is dense, and some detectors store the opposite; if the optimiser goes nowhere, check that your radiograph and a rendered DRR have the same polarity before blaming the optimiser.

In [ ]:
import shlex
import subprocess

VOLUME = WORK / "ct.nii.gz"        # your CT
XRAY = WORK / "xray.dcm"           # your radiograph
OUTDIR = WORK / "registration"
OUTDIR.mkdir(exist_ok=True)

SDD = 1020.0     # source to detector, mm
DELX = 0.194     # detector pixel spacing, mm
HEIGHT = 1536    # image height, pixels

# A starting pose in roughly the right neighbourhood: rotations in degrees,
# translations in mm, in the convention xvr prints under --help.
INIT_ROT = (0.0, 0.0, 0.0)
INIT_XYZ = (0.0, 0.0, 1020.0)

command = f"""
xvr register {shlex.quote(str(XRAY))}
  -v {shlex.quote(str(VOLUME))}
  -o {shlex.quote(str(OUTDIR))}
  --sdd {SDD} --delx {DELX} --height {HEIGHT}
  --rot {INIT_ROT[0]} {INIT_ROT[1]} {INIT_ROT[2]}
  --xyz {INIT_XYZ[0]} {INIT_XYZ[1]} {INIT_XYZ[2]}
"""
command = " ".join(command.split())
print(command, "\n")

result = subprocess.run(command, shell=True, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("--- failed, stderr follows ---")
    print(result.stderr)

## 7. Train a patient-specific network first

This is the part the browser demo cannot do, and the reason the paper exists. `xvr train` learns a pose regressor from your CT alone, by rendering it from thousands of sampled poses. A handful of minutes on a T4. Afterwards registration needs no initial guess from you: the network supplies one and the optimiser refines it.

Read the flags first, for the same reason as before.

In [ ]:
!xvr train --help

## 8. Look at the result

Registration is only believable if you look at it. Overlay the radiograph and a DRR rendered at the recovered pose: where the two agree the overlay goes grey, and where they do not you get colour fringes, exactly as in the browser demo.

This lists what `xvr` wrote, and gives you the overlay function to plot with.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for f in sorted(OUTDIR.rglob("*")):
    if f.is_file():
        print(f.relative_to(OUTDIR), f"{f.stat().st_size / 1e3:.0f} kB")


def overlay(fixed, moving, title="fixed in red, moving in cyan, grey where they agree"):
    """Both arrays are 2D. Each is windowed to its own 1st and 99.5th percentile."""

    def window(a):
        a = np.asarray(a, dtype=np.float32)
        lo, hi = np.percentile(a, [1.0, 99.5])
        return np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)

    rgb = np.stack([window(fixed), window(moving), window(moving)], axis=-1)
    plt.figure(figsize=(6, 6))
    plt.imshow(rgb)
    plt.title(title, fontsize=9)
    plt.axis("off")
    plt.show()


# overlay(fixed_array, drr_array)

## Where to go next

- Full CLI documentation: [xvr.csail.mit.edu](https://xvr.csail.mit.edu/)
- Pretrained models and the datasets from the paper, on HuggingFace: `eigenvivek/xvr` and `eigenvivek/xvr-data`
- The renderer on its own, to build something else on the forward model: [`nanodrr`](https://github.com/eigenvivek/nanodrr)
- The browser version of the optimisation step, with no install: [vanioantunes.com/3d/xvr/](https://vanioantunes.com/3d/xvr/)

Notebook by Vanio Antunes. `xvr` and `nanodrr` are by Vivek Gopalakrishnan and colleagues, MIT licensed; this notebook is a wrapper around them and claims no part of that work.